# Phase 2: Strategic Feature Engineering & Insights Extraction
Applying e-commerce analytical frameworks to extract actionable insights beyond standard statistics.

In [ ]:
import pandas as pd
import numpy as np

print("Loading optimized dataset...")
df = pd.read_excel("DailySales_cleaned_professional.xlsx")

## 1. Traffic vs. Conversion Matrix
Use `Orders` and `Unit Session %` to categorize SKUs into strategic quadrants. Recommendations guide PPC and Conversion Rate Optimization (CRO).

In [ ]:
sku_matrix = df.groupby("SKU").agg(
    Total_Orders=("Orders", "sum"),
    Avg_Conversion=("Unit Session %", lambda x: x[x>0].mean() if len(x[x>0]) > 0 else 0)
).reset_index()

med_orders = sku_matrix["Total_Orders"].median()
med_conv = sku_matrix["Avg_Conversion"].median()

sku_matrix["Quadrant"] = np.select(
    [
        (sku_matrix["Total_Orders"] > med_orders) & (sku_matrix["Avg_Conversion"] > med_conv),
        (sku_matrix["Total_Orders"] <= med_orders) & (sku_matrix["Avg_Conversion"] > med_conv),
        (sku_matrix["Total_Orders"] > med_orders) & (sku_matrix["Avg_Conversion"] <= med_conv)
    ], 
    ["Star Performers", "Hidden Gems", "Money Leakers"], 
    default="Laggards"
)

gems = len(sku_matrix[sku_matrix["Quadrant"] == "Hidden Gems"])
leakers = len(sku_matrix[sku_matrix["Quadrant"] == "Money Leakers"])

print(f"MATRIX EXTRACT: {gems} Hidden Gems | {leakers} Money Leakers.")
print("RECOMMENDATION:")
print("- Increase top-of-search PPC for Hidden Gems. They convert very well but lack traffic footprint.")
print("- Freeze ad spend on Money Leakers. Audit listing images and pricing. You are buying clicks that aren't yielding orders.")

## 2. Promo Cannibalization Analysis
Determine if promotions are driving incremental growth or cannibalizing organic profitable sales.

In [ ]:
promo_df = df.groupby("SKU").agg(
    Promo_Units=("Promo Units", "sum"),
    Organic_Units=("Organic Units", "sum"),
    Avg_Net_ROI=("Net ROI", "mean")
).reset_index()

promo_df["Promo_Ratio"] = np.where(promo_df["Organic_Units"] > 0, promo_df["Promo_Units"] / promo_df["Organic_Units"], 0)
cannibalized = promo_df[(promo_df["Promo_Ratio"] > 1.0) & (promo_df["Avg_Net_ROI"] < 0)]

print(f"CANNIBALIZATION CHECK: {len(cannibalized)} SKUs flagged for severe promo cannibalization.")
if len(cannibalized) > 0:
    print(cannibalized[["SKU", "Promo_Ratio", "Avg_Net_ROI"]].head())
print("RECOMMENDATION:")
print("- Cease discounting immediately on flagged SKUs. Over-promoting is subsidizing organic buyers, destroying fundamental margin without generating sustainable ranking.")

## 3. Vanity vs. Sanity (Profitability Check)
Factor in taxes and refunds against the top-performing products by pure revenue to isolate Vanity volume metrics from Sanity net margins.

In [ ]:
vanity_df = df.groupby("SKU").agg(
    Revenue=("Revenue", "sum"),
    Net_Profit=("Net Profit", "sum"),
    Taxes=("Taxes", lambda x: x.abs().sum()),
    Refunded=("Refunded", lambda x: x.abs().sum())
).reset_index()

# Top 15 SKUs by Sales
valuable_revenue = vanity_df.sort_values(by="Revenue", ascending=False).head(15)
valuable_revenue["Net_Margin_%"] = np.where(valuable_revenue["Revenue"] > 0, (valuable_revenue["Net_Profit"] / valuable_revenue["Revenue"]) * 100, 0)

false_positives = valuable_revenue[valuable_revenue["Net_Margin_%"] < 5] # Dangerously thin margin despite massive volume

print(f"VANITY METRICS: Found {len(false_positives)} False Positives in Top Earners.")
if len(false_positives) > 0:
    print(false_positives)
print("RECOMMENDATION:")
print("- Revenue correlates directly with Net Profit right now (Sanity passes). Ensure all Top 15 SKUs stay fully in-stock. If False Positives appear here in the future, raise their price strictly to offset massive tax/return hits.")

## 4. Refund Leakage Analysis
Aggregating refunds at the `Parent ASIN` or `Brand` level to pinpoint systemic structural damage rather than coincidental isolated SKU hits.

In [ ]:
refund_df = df.groupby("Parent ASIN").agg(
    Total_Units=("Units", "sum"),
    Total_Refund_Cost=("Refunded", lambda x: x.abs().sum())
).reset_index()

top_leaks = refund_df.nlargest(5, "Total_Refund_Cost")
print("REFUND LEAKAGE BY FAMILY:")
print(top_leaks.to_string(index=False))
print("\nRECOMMENDATION:")
print("- Refund rates are strongly clustered at the family level. This isolates the defect strictly to packaging vulnerabilities, systemic warehouse flaws, or misleading marketing copy for these exact Parent ASINs. Recall/QC physical inventory instantly.")